# NLP from TF-IDF to Transformers: Interview-Ready Guide

This notebook covers the full NLP interview spectrum:

1. **Text Preprocessing** — tokenization, stemming, lemmatization, TF-IDF, Word2Vec
2. **Classical NLP Tasks** — classification, sentiment, NER, LDA topic modeling
3. **Attention Mechanism** — self-attention from scratch, heatmaps, positional encoding
4. **Pretrained Models** — BERT, sentence-transformers, fine-tuning, zero-shot
5. **15 NLP Interview Q&A** — plain English + code proof

**Datasets used:** 20 Newsgroups (sklearn), synthetic sentiment data

---
*Run cells top-to-bottom. All imports are handled with graceful fallbacks.*

In [ ]:
# Core imports — these should always be available
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

from collections import Counter
import re
import math
import json

print('Core imports OK')
print(f'numpy {np.__version__}')
print(f'matplotlib {matplotlib.__version__}')

In [ ]:
# sklearn imports
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_score, recall_score, f1_score, accuracy_score
)
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

print('sklearn imports OK')

---
## SECTION 1 — Text Preprocessing

Every NLP interview starts here. The fundamentals distinguish candidates who understand NLP from those who just call `.fit()`.

### 1.1 Tokenization

**Tokenization** splits raw text into meaningful units (tokens). Tokens can be words, subwords, sentences, or characters.

- **Word tokenization**: split on whitespace/punctuation
- **Subword tokenization** (BPE, WordPiece): handles unknown words — used in BERT, GPT
- **Sentence tokenization**: split into sentences first

Interview tip: Know *why* subword tokenization was invented — it solves the out-of-vocabulary problem.

In [ ]:
# Simple word tokenizer from scratch
def simple_tokenize(text):
    """Lowercase, remove punctuation, split on whitespace."""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    tokens = text.split()
    return tokens

# Sentence tokenizer
def sentence_tokenize(text):
    """Split on sentence-ending punctuation."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s for s in sentences if s]

sample = "Machine learning is fascinating! I love NLP. Transformers changed everything in 2017."

word_tokens = simple_tokenize(sample)
sentences = sentence_tokenize(sample)

print('=== Word Tokenization ===')
print(word_tokens)
print(f'Token count: {len(word_tokens)}')

print('\n=== Sentence Tokenization ===')
for i, s in enumerate(sentences):
    print(f'  [{i}]: {s}')

### 1.2 Stemming vs. Lemmatization

| Method | Output | Speed | Accuracy | Example |
|---|---|---|---|---|
| Stemming | Word stem (may not be real word) | Fast | Lower | "running" -> "run" |
| Lemmatization | Dictionary base form | Slower | Higher | "better" -> "good" |

**Interview answer**: Use lemmatization for tasks requiring semantic meaning. Stemming is fine for IR/search where speed matters.

In [ ]:
# Stemming from scratch — Porter Stemmer (simplified)
def simple_stem(word):
    """Very simplified stemmer — strip common suffixes."""
    word = word.lower()
    suffixes = ['ing', 'tion', 'ness', 'ment', 'ly', 'ed', 'er', 'est', 's']
    for suffix in sorted(suffixes, key=len, reverse=True):
        if word.endswith(suffix) and len(word) - len(suffix) >= 3:
            return word[:-len(suffix)]
    return word

# Lemmatization — simplified rule-based
lemma_map = {
    'running': 'run', 'ran': 'run', 'runs': 'run',
    'better': 'good', 'best': 'good',
    'worse': 'bad', 'worst': 'bad',
    'children': 'child', 'mice': 'mouse', 'feet': 'foot',
    'was': 'be', 'were': 'be', 'is': 'be', 'are': 'be'
}

def simple_lemmatize(word):
    return lemma_map.get(word.lower(), word.lower())

test_words = ['running', 'classification', 'happiness', 'better',
              'children', 'was', 'cats', 'tokenization']

print(f"{'Word':<20} {'Stemmed':<20} {'Lemmatized':<20}")
print('-' * 60)
for w in test_words:
    stem = simple_stem(w)
    lemma = simple_lemmatize(w)
    print(f"{w:<20} {stem:<20} {lemma:<20}")

print('\nNote: Production code uses NLTK PorterStemmer / WordNetLemmatizer or spaCy')

### 1.3 Stop Words — When to Remove and When NOT To

**Stop words** are high-frequency words with little semantic content: *the, a, is, in, of*.

**Remove stop words when:**
- Building a search index or TF-IDF features
- Topic modeling (LDA)
- Document similarity comparisons

**Keep stop words when:**
- Sentiment analysis ("not good" vs "good" — 'not' matters!)
- Named entity recognition
- Machine translation
- Any task where syntax/negation matters

**Interview trap**: Automatically removing stop words for sentiment analysis is a common mistake.

In [ ]:
STOP_WORDS = {
    'i', 'me', 'my', 'myself', 'we', 'our', 'you', 'your', 'he', 'him',
    'she', 'her', 'it', 'its', 'they', 'them', 'what', 'which', 'who',
    'this', 'that', 'these', 'those', 'am', 'is', 'are', 'was', 'were',
    'be', 'been', 'being', 'have', 'has', 'had', 'do', 'does', 'did',
    'will', 'would', 'could', 'should', 'may', 'might', 'shall', 'can',
    'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until',
    'of', 'at', 'by', 'for', 'with', 'about', 'between', 'through',
    'in', 'out', 'on', 'off', 'over', 'under', 'then', 'once', 'to',
    'from', 'up', 'down', 'into', 'during', 'before', 'after'
}

def remove_stop_words(tokens, stop_words=STOP_WORDS):
    return [t for t in tokens if t not in stop_words]

# Demonstrate the negation problem
sentences_demo = [
    "This movie is not good at all",
    "The product is really amazing",
    "I do not like this restaurant"
]

print('=== Why "not" matters in sentiment analysis ===')
for sent in sentences_demo:
    tokens = simple_tokenize(sent)
    filtered = remove_stop_words(tokens)
    print(f'\nOriginal : {tokens}')
    print(f'Filtered : {filtered}')
    if 'not' in tokens:
        print('  WARNING: "not" was removed — sentiment flipped!')

### 1.4 TF-IDF from Scratch

TF-IDF measures how important a word is to a document in a corpus.

```
TF(t, d)  = count(t in d) / count(all words in d)
IDF(t, D) = log( (1 + N) / (1 + df(t)) ) + 1   # sklearn's smooth variant
TF-IDF    = TF * IDF
```

**Key insight**: IDF *downweights* common words (low information) and *upweights* rare words (high information).

**Interview question**: Why does TF-IDF work better than raw word counts? Because raw counts favor long documents and common words.

In [ ]:
def compute_tf(tokens):
    """Term Frequency: proportion of each token in the document."""
    count = Counter(tokens)
    total = len(tokens)
    return {word: cnt / total for word, cnt in count.items()}

def compute_idf(documents_tokens):
    """Inverse Document Frequency (smooth variant, matching sklearn)."""
    N = len(documents_tokens)
    df = Counter()
    for tokens in documents_tokens:
        unique_tokens = set(tokens)
        for token in unique_tokens:
            df[token] += 1
    idf = {}
    for word, freq in df.items():
        idf[word] = math.log((1 + N) / (1 + freq)) + 1
    return idf

def compute_tfidf(tokens, idf):
    """Compute TF-IDF for a single document."""
    tf = compute_tf(tokens)
    return {word: tf_val * idf.get(word, 0) for word, tf_val in tf.items()}

# Toy corpus
corpus = [
    "machine learning is a subset of artificial intelligence",
    "deep learning uses neural networks with multiple layers",
    "natural language processing is a branch of machine learning",
    "transformers revolutionized natural language processing in 2017"
]

corpus_tokens = [simple_tokenize(doc) for doc in corpus]
idf_scores = compute_idf(corpus_tokens)

print('=== IDF Scores (lower = more common = less informative) ===')
for word in ['machine', 'learning', 'natural', 'language', 'transformers', 'is', 'a']:
    if word in idf_scores:
        print(f'  {word:<20}: IDF = {idf_scores[word]:.4f}')

print('\n=== TF-IDF for document 0 (top 5 terms) ===')
tfidf_doc0 = compute_tfidf(corpus_tokens[0], idf_scores)
top5 = sorted(tfidf_doc0.items(), key=lambda x: x[1], reverse=True)[:5]
for word, score in top5:
    print(f'  {word:<20}: TF-IDF = {score:.4f}')

In [ ]:
# Verify with sklearn TF-IDF
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(corpus)
feature_names = vectorizer.get_feature_names_out()

print('=== sklearn TF-IDF for document 0 (top 5 terms) ===')
doc0_scores = X[0].toarray()[0]
top5_idx = doc0_scores.argsort()[-5:][::-1]
for idx in top5_idx:
    print(f'  {feature_names[idx]:<20}: TF-IDF = {doc0_scores[idx]:.4f}')

print(f'\nMatrix shape: {X.shape} (4 docs x {X.shape[1]} unique terms)')
print('Our scratch implementation and sklearn give the same ranking. ')

### 1.5 Word2Vec Intuition — What Embeddings Actually Capture

Word2Vec learns dense vector representations where **similar words cluster together**.

**Two architectures:**
- **CBOW** (Continuous Bag of Words): predict center word from context
- **Skip-gram**: predict context words from center word

**What vectors capture:**
- Semantic similarity: king - man + woman ≈ queen
- Syntactic patterns: walk - walked ≈ swim - swam
- Domain relationships

**Key insight**: The network is a "fake" task. We don't care about predictions — we want the *hidden layer weights* as embeddings.

In [ ]:
# Demonstrate the analogy arithmetic idea with a tiny hand-crafted embedding
# (Real Word2Vec needs large corpora — we illustrate the concept)

np.random.seed(42)

# Toy 3D word vectors (hand-crafted to illustrate semantic directions)
word_vectors = {
    'king':   np.array([0.99, 0.02, 0.70]),  # royalty=high, female=low, power=high
    'queen':  np.array([0.99, 0.98, 0.70]),  # royalty=high, female=high, power=high
    'man':    np.array([0.05, 0.02, 0.50]),  # royalty=low, female=low, power=medium
    'woman':  np.array([0.05, 0.98, 0.50]),  # royalty=low, female=high, power=medium
    'prince': np.array([0.80, 0.02, 0.60]),
    'princess': np.array([0.80, 0.98, 0.60]),
    'dog':    np.array([0.01, 0.01, 0.10]),
    'cat':    np.array([0.01, 0.01, 0.12]),
}

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)

def analogy(word_a, word_b, word_c, vectors):
    """word_a is to word_b as word_c is to ?"""
    target = vectors[word_b] - vectors[word_a] + vectors[word_c]
    exclude = {word_a, word_b, word_c}
    scores = {
        w: cosine_similarity(target, v)
        for w, v in vectors.items() if w not in exclude
    }
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:3]

print('=== Word Analogy: king - man + woman = ? ===')
result = analogy('man', 'king', 'woman', word_vectors)
for word, score in result:
    print(f'  {word:<15}: similarity = {score:.4f}')

print('\n=== Cosine Similarity Matrix (semantic neighbors) ===')
words_to_compare = ['king', 'queen', 'man', 'woman', 'dog', 'cat']
print(f"{'':10}" + ''.join(f'{w:9}' for w in words_to_compare))
for w1 in words_to_compare:
    row = f'{w1:10}'
    for w2 in words_to_compare:
        sim = cosine_similarity(word_vectors[w1], word_vectors[w2])
        row += f'{sim:9.2f}'
    print(row)

---
## SECTION 2 — Classical NLP Tasks

These are the bread-and-butter tasks you will implement in most NLP interviews.

### 2.1 Loading the 20 Newsgroups Dataset

In [ ]:
# Use a subset of 4 categories for speed
categories = [
    'rec.sport.hockey',
    'sci.space',
    'comp.graphics',
    'talk.politics.guns'
]

print('Loading 20 Newsgroups dataset (4 categories)...')
newsgroups_train = fetch_20newsgroups(
    subset='train',
    categories=categories,
    remove=('headers', 'footers', 'quotes'),
    random_state=42
)
newsgroups_test = fetch_20newsgroups(
    subset='test',
    categories=categories,
    remove=('headers', 'footers', 'quotes'),
    random_state=42
)

print(f'Training samples: {len(newsgroups_train.data)}')
print(f'Test samples    : {len(newsgroups_test.data)}')
print(f'Categories      : {newsgroups_train.target_names}')
print('\n--- Sample document (first 200 chars) ---')
print(newsgroups_train.data[0][:200])
print(f'\nLabel: {newsgroups_train.target_names[newsgroups_train.target[0]]}')

### 2.2 Text Classification with TF-IDF + Logistic Regression

This pipeline is a strong baseline for many production NLP tasks:
- TF-IDF converts text to a sparse numeric matrix
- Logistic Regression learns a linear boundary in TF-IDF space

**Interview answer**: "Always start with a TF-IDF + LR baseline. It's fast, interpretable, and often within 5% of deep learning on short texts."

In [ ]:
# Build the pipeline
text_clf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2),  # unigrams + bigrams
        sublinear_tf=True,   # apply log(1+tf) smoothing
        stop_words='english'
    )),
    ('clf', LogisticRegression(max_iter=1000, C=1.0, random_state=42))
])

text_clf_pipeline.fit(newsgroups_train.data, newsgroups_train.target)

y_pred = text_clf_pipeline.predict(newsgroups_test.data)
y_true = newsgroups_test.target

print('=== TF-IDF + Logistic Regression Results ===')
print(f'Accuracy: {accuracy_score(y_true, y_pred):.4f}')
print()
print(classification_report(
    y_true, y_pred,
    target_names=newsgroups_test.target_names
))

In [ ]:
# Visualize confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_true, y_pred)
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)
ax.set(xticks=np.arange(cm.shape[1]),
       yticks=np.arange(cm.shape[0]),
       xticklabels=newsgroups_test.target_names,
       yticklabels=newsgroups_test.target_names,
       ylabel='True label',
       xlabel='Predicted label',
       title='Confusion Matrix — TF-IDF + Logistic Regression')
plt.setp(ax.get_xticklabels(), rotation=20, ha='right', rotation_mode='anchor')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=80, bbox_inches='tight')
plt.show()
print('Confusion matrix saved.')

### 2.3 Sentiment Analysis

We build a simple sentiment analyzer using TF-IDF features on synthetic reviews. In practice you would use labeled datasets like IMDb, SST-2, or Yelp.

In [ ]:
# Synthetic sentiment dataset
positive_reviews = [
    "This product is absolutely amazing and I love it",
    "Great quality, fast shipping, highly recommend",
    "Exceeded my expectations, will buy again",
    "Fantastic customer service and wonderful product",
    "Best purchase I have ever made, very satisfied",
    "Incredible value for money, top notch quality",
    "Outstanding performance and excellent build quality",
    "Very happy with this product, works perfectly",
    "Superb design and very easy to use",
    "Brilliant product, exactly what I needed",
    "Love it so much, works great every time",
    "Perfect product, zero complaints, five stars"
]
negative_reviews = [
    "Terrible product, broke after two days",
    "Worst purchase ever, complete waste of money",
    "Very disappointed, does not work as described",
    "Poor quality and horrible customer service",
    "Would not recommend this to anyone",
    "Defective product arrived, packaging was damaged",
    "Returned it immediately, completely useless",
    "Do not buy this, total scam and waste",
    "Absolutely awful, nothing like the description",
    "Broken on arrival and support did not help",
    "Very poor quality, fell apart within a week",
    "Garbage product, do not waste your money"
]
neutral_reviews = [
    "It is okay, not great but does the job",
    "Average product, nothing special about it",
    "Decent quality but a bit overpriced",
    "Works as expected, nothing more nothing less",
    "Mediocre, some good features some bad ones",
    "Not what I expected but it still functions"
]

reviews = positive_reviews + negative_reviews + neutral_reviews
labels = (['positive'] * len(positive_reviews) +
          ['negative'] * len(negative_reviews) +
          ['neutral'] * len(neutral_reviews))
label_map = {'positive': 1, 'negative': 0, 'neutral': 2}
labels_num = [label_map[l] for l in labels]

X_train, X_test, y_train, y_test = train_test_split(
    reviews, labels_num, test_size=0.3, random_state=42, stratify=labels_num
)

sentiment_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=500)),
    ('clf', LogisticRegression(max_iter=500, C=5.0, random_state=42))
])
sentiment_pipeline.fit(X_train, y_train)
y_pred_sent = sentiment_pipeline.predict(X_test)

print('=== Sentiment Analysis Results ===')
id2label = {v: k for k, v in label_map.items()}
print(classification_report(y_test, y_pred_sent,
      target_names=['negative', 'positive', 'neutral']))

# Test on custom examples
test_sentences = [
    "This is absolutely wonderful, I love it!",
    "Complete garbage, do not buy this product",
    "It is okay I suppose",
    "Not bad but not great either",
    "Phenomenal quality, best product ever"
]
print('\n=== Custom Prediction ===')
for s in test_sentences:
    pred = sentiment_pipeline.predict([s])[0]
    proba = sentiment_pipeline.predict_proba([s])[0]
    label = id2label[pred]
    conf = max(proba)
    print(f'  [{label:10s} {conf:.0%}] {s}')

### 2.4 Named Entity Recognition (NER) — Concept

NER identifies real-world entities in text: people, organizations, locations, dates.

**How classical NER works:**
- Rule-based: regex patterns for dates, postal codes
- Feature-based: capitalization, POS tags, word context -> CRF
- Deep learning: BiLSTM-CRF, then BERT-based NER

**Interview Q**: What features would you use for NER?
- Is word capitalized? (proper noun signal)
- Previous/next words
- POS tag
- Is it in a known gazetteer (list of known entities)?
- Word shape: "Xxxx" (capitalized), "XXXX" (all caps), "dddd" (year)

In [ ]:
# Rule-based NER demo — simple regex patterns
import re

def simple_ner(text):
    """Very simple rule-based NER using regex patterns."""
    entities = []

    # Dates: Month DD, YYYY or YYYY-MM-DD
    for match in re.finditer(
        r'\b(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},\s*\d{4}\b',
        text
    ):
        entities.append(('DATE', match.group(), match.span()))

    for match in re.finditer(r'\b\d{4}-\d{2}-\d{2}\b', text):
        entities.append(('DATE', match.group(), match.span()))

    # Money: $1,234 or $12.50
    for match in re.finditer(r'\$[\d,]+(?:\.\d{2})?', text):
        entities.append(('MONEY', match.group(), match.span()))

    # Email addresses
    for match in re.finditer(r'\b[\w.-]+@[\w.-]+\.\w{2,}\b', text):
        entities.append(('EMAIL', match.group(), match.span()))

    # URLs
    for match in re.finditer(r'https?://\S+', text):
        entities.append(('URL', match.group(), match.span()))

    # Capitalized sequences (potential proper nouns) — heuristic
    for match in re.finditer(r'\b([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)\b', text):
        if not any(match.start() >= s and match.end() <= e for _, _, (s, e) in entities):
            entities.append(('PERSON/ORG', match.group(), match.span()))

    return sorted(entities, key=lambda x: x[2][0])

ner_sample = (
    "On March 15, 2023, John Smith met Sarah Connor at Google headquarters. "
    "The deal was worth $1,500,000 and closed on 2023-03-20. "
    "Contact us at info@company.com or visit https://example.com"
)

print('=== Simple Rule-Based NER ===')
print(f'Input: {ner_sample}')
print('\nEntities found:')
for etype, etext, (start, end) in simple_ner(ner_sample):
    print(f'  [{etype:12s}] "{etext}" at position {start}-{end}')

print('\nNote: Production NER uses spaCy, Hugging Face, or AWS Comprehend')

### 2.5 Topic Modeling with LDA

**Latent Dirichlet Allocation (LDA)** is an unsupervised technique that discovers hidden topics in a corpus.

**Key assumptions:**
- Each document is a mixture of topics
- Each topic is a distribution over words
- "Latent" = topics are hidden variables we infer

**Interview Q**: How does LDA differ from k-means clustering?
- k-means: hard assignment (one cluster per doc)
- LDA: soft assignment (document can be 30% Topic A, 70% Topic B)

In [ ]:
print('Running LDA on 20 Newsgroups (4 categories, looking for 4 topics)...')

# Use the training data we already loaded
cv = CountVectorizer(
    max_features=2000,
    stop_words='english',
    min_df=5,
    max_df=0.95
)
X_counts = cv.fit_transform(newsgroups_train.data)

lda = LatentDirichletAllocation(
    n_components=4,
    max_iter=20,
    learning_method='online',
    random_state=42
)
lda.fit(X_counts)

feature_names_lda = cv.get_feature_names_out()

print('=== Discovered Topics (top 10 words each) ===')
for topic_idx, topic in enumerate(lda.components_):
    top_words_idx = topic.argsort()[-10:][::-1]
    top_words = [feature_names_lda[i] for i in top_words_idx]
    print(f'\nTopic {topic_idx}: {" | ".join(top_words)}')

print('\n--- Interpreting topics ---')
print('Can you match each topic to: hockey, space, graphics, politics?')

In [ ]:
# Visualize topic-word distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for topic_idx, (topic, ax) in enumerate(zip(lda.components_, axes)):
    top_n = 10
    top_idx = topic.argsort()[-top_n:][::-1]
    top_words = [feature_names_lda[i] for i in top_idx]
    top_weights = [topic[i] for i in top_idx]

    bars = ax.barh(range(top_n), top_weights[::-1], color=f'C{topic_idx}', alpha=0.8)
    ax.set_yticks(range(top_n))
    ax.set_yticklabels(top_words[::-1])
    ax.set_title(f'Topic {topic_idx} — Top Words', fontsize=12, fontweight='bold')
    ax.set_xlabel('Word Weight')
    ax.invert_xaxis()
    ax.yaxis.set_label_position('right')
    ax.yaxis.tick_right()

plt.suptitle('LDA Topics Discovered from 20 Newsgroups', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('lda_topics.png', dpi=80, bbox_inches='tight')
plt.show()
print('LDA topic visualization saved.')

### 2.6 Evaluation Metrics for Text Classification

Understanding precision/recall/F1 is fundamental to any ML interview.

```
Precision = TP / (TP + FP)  -- of all predicted positive, how many are actually positive?
Recall    = TP / (TP + FN)  -- of all actual positive, how many did we find?
F1        = 2 * P * R / (P + R)  -- harmonic mean, penalizes extreme imbalance
```

**When to use which:**
- High precision: spam detection (don't block real emails)
- High recall: cancer screening (don't miss actual cancer)
- F1: when you need to balance both

In [ ]:
# Illustrate precision/recall tradeoff from scratch
def compute_metrics_from_scratch(y_true, y_pred, pos_label=1):
    tp = sum(1 for t, p in zip(y_true, y_pred) if t == pos_label and p == pos_label)
    fp = sum(1 for t, p in zip(y_true, y_pred) if t != pos_label and p == pos_label)
    fn = sum(1 for t, p in zip(y_true, y_pred) if t == pos_label and p != pos_label)
    tn = sum(1 for t, p in zip(y_true, y_pred) if t != pos_label and p != pos_label)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy  = (tp + tn) / (tp + fp + fn + tn)

    return {'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
            'Precision': precision, 'Recall': recall,
            'F1': f1, 'Accuracy': accuracy}

# Simulated predictions for spam detection
np.random.seed(42)
y_true_spam = np.array([1]*50 + [0]*150)  # 50 spam, 150 ham
np.random.shuffle(y_true_spam)

# Conservative model: misses some spam but never marks ham as spam
y_pred_conservative = y_true_spam.copy()
y_pred_conservative[y_true_spam == 1] = np.where(
    np.random.rand(50) > 0.4, 1, 0)  # misses 40% of spam

# Aggressive model: catches all spam but some false positives
y_pred_aggressive = y_true_spam.copy()
y_pred_aggressive[y_true_spam == 0] = np.where(
    np.random.rand(150) > 0.85, 1, 0)  # flags 15% of ham

print('=== Precision/Recall Tradeoff Demo ===')
for name, preds in [('Conservative', y_pred_conservative),
                     ('Aggressive',   y_pred_aggressive)]:
    m = compute_metrics_from_scratch(y_true_spam, preds, pos_label=1)
    print(f'\n{name} model:')
    print(f"  Precision: {m['Precision']:.3f}  (of predicted spam, {m['Precision']:.0%} truly spam)")
    print(f"  Recall   : {m['Recall']:.3f}  (caught {m['Recall']:.0%} of all actual spam)")
    print(f"  F1 Score : {m['F1']:.3f}")
    print(f"  Accuracy : {m['Accuracy']:.3f}")
    print(f"  TP={m['TP']} FP={m['FP']} FN={m['FN']} TN={m['TN']}")

---
## SECTION 3 — Attention Mechanism

Attention is the core innovation behind Transformers. If you understand attention, you understand 90% of modern NLP.

### 3.1 Self-Attention from Scratch

Self-attention lets each token in a sequence attend to every other token.

**The three matrices:**
- **Q** (Query): what this token is *looking for*
- **K** (Key): what this token *offers* to others
- **V** (Value): the actual *content* passed if attended to

**Formula:**
```
Attention(Q, K, V) = softmax(Q @ K.T / sqrt(d_k)) @ V
```

**Why divide by sqrt(d_k)?** To prevent dot products from growing too large, causing vanishingly small softmax gradients.

In [ ]:
def softmax(x, axis=-1):
    """Numerically stable softmax."""
    x_shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x_shifted)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Self-attention mechanism.
    Q: (seq_len, d_k)
    K: (seq_len, d_k)
    V: (seq_len, d_v)
    Returns: (seq_len, d_v), (seq_len, seq_len)
    """
    d_k = Q.shape[-1]
    # Dot product similarity scores
    scores = Q @ K.T / np.sqrt(d_k)      # (seq_len, seq_len)

    if mask is not None:
        scores = scores + mask * -1e9     # mask out padding or future tokens

    attention_weights = softmax(scores)   # (seq_len, seq_len)
    output = attention_weights @ V        # (seq_len, d_v)
    return output, attention_weights

# Demo: 5-token sequence, d_model=8, d_k=4
np.random.seed(42)
seq_len = 5
d_model = 8
d_k = 4
d_v = 4

# Token embeddings (normally learned, here random)
token_embeddings = np.random.randn(seq_len, d_model)

# Projection matrices (normally learned)
W_Q = np.random.randn(d_model, d_k) * 0.1
W_K = np.random.randn(d_model, d_k) * 0.1
W_V = np.random.randn(d_model, d_v) * 0.1

# Project to Q, K, V
Q = token_embeddings @ W_Q   # (5, 4)
K = token_embeddings @ W_K   # (5, 4)
V = token_embeddings @ W_V   # (5, 4)

output, attn_weights = scaled_dot_product_attention(Q, K, V)

print('=== Self-Attention Shapes ===')
print(f'Input embeddings : {token_embeddings.shape}')
print(f'Q, K matrices    : {Q.shape}')
print(f'V matrix         : {V.shape}')
print(f'Attention weights: {attn_weights.shape}  (each row sums to 1)')
print(f'Output           : {output.shape}')
print()
print('Attention weight matrix (each row = how token i attends to all others):')
print(np.round(attn_weights, 3))
print(f'\nRow sums: {np.round(attn_weights.sum(axis=1), 4)}  (should all be 1.0)')

In [ ]:
# Semantic attention heatmap with interpretable tokens
words = ['The', 'cat', 'sat', 'on', 'mat']

# Craft embeddings with semantic meaning
# [noun, verb, preposition, article, position]
token_emb_semantic = np.array([
    [0.1, 0.0, 0.0, 0.9, 0.1],  # The  -- article
    [0.9, 0.0, 0.0, 0.0, 0.5],  # cat  -- noun
    [0.0, 0.9, 0.0, 0.0, 0.3],  # sat  -- verb
    [0.0, 0.0, 0.9, 0.0, 0.2],  # on   -- preposition
    [0.9, 0.0, 0.0, 0.1, 0.0],  # mat  -- noun
])

np.random.seed(7)
W_Q2 = np.random.randn(5, 3) * 0.5
W_K2 = np.random.randn(5, 3) * 0.5
W_V2 = np.random.randn(5, 3) * 0.5

Q2 = token_emb_semantic @ W_Q2
K2 = token_emb_semantic @ W_K2
V2 = token_emb_semantic @ W_V2

_, attn2 = scaled_dot_product_attention(Q2, K2, V2)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(attn2, cmap='YlOrRd', vmin=0, vmax=attn2.max())
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Attention Weight', rotation=270, labelpad=15)
ax.set_xticks(range(len(words)))
ax.set_yticks(range(len(words)))
ax.set_xticklabels(words, fontsize=12)
ax.set_yticklabels(words, fontsize=12)
ax.set_xlabel('Key (attended TO)', fontsize=12)
ax.set_ylabel('Query (attending FROM)', fontsize=12)
ax.set_title('Self-Attention Heatmap\n"The cat sat on the mat"', fontsize=13)
for i in range(len(words)):
    for j in range(len(words)):
        ax.text(j, i, f'{attn2[i,j]:.2f}', ha='center', va='center',
                fontsize=9, color='black')
plt.tight_layout()
plt.savefig('attention_heatmap.png', dpi=80, bbox_inches='tight')
plt.show()
print('Attention heatmap saved.')

### 3.2 Multi-Head Attention Intuition

Instead of one attention head, use **h heads** in parallel, each learning different relationship types:
- Head 1 might learn syntactic dependencies (subject-verb)
- Head 2 might learn co-reference (pronoun-noun)
- Head 3 might learn positional relationships

**Formula:**
```
MultiHead(Q,K,V) = Concat(head_1, ..., head_h) @ W_O
where head_i = Attention(Q @ W_Q_i, K @ W_K_i, V @ W_V_i)
```

In [ ]:
def multi_head_attention(X, num_heads, d_model):
    """
    X: (seq_len, d_model)
    Returns output: (seq_len, d_model), list of attention weight matrices
    """
    assert d_model % num_heads == 0
    d_head = d_model // num_heads
    seq_len = X.shape[0]

    all_heads = []
    all_attn_weights = []

    for h in range(num_heads):
        np.random.seed(h * 13 + 7)
        W_Q_h = np.random.randn(d_model, d_head) * 0.1
        W_K_h = np.random.randn(d_model, d_head) * 0.1
        W_V_h = np.random.randn(d_model, d_head) * 0.1

        Q_h = X @ W_Q_h
        K_h = X @ W_K_h
        V_h = X @ W_V_h

        head_out, attn_w = scaled_dot_product_attention(Q_h, K_h, V_h)
        all_heads.append(head_out)
        all_attn_weights.append(attn_w)

    # Concatenate and project
    concat = np.concatenate(all_heads, axis=-1)  # (seq_len, d_model)
    np.random.seed(99)
    W_O = np.random.randn(d_model, d_model) * 0.1
    output = concat @ W_O

    return output, all_attn_weights

num_heads = 4
d_model_mha = 8

np.random.seed(42)
X_mha = np.random.randn(5, d_model_mha)

mha_output, mha_weights = multi_head_attention(X_mha, num_heads, d_model_mha)

print('=== Multi-Head Attention ===')
print(f'Input shape   : {X_mha.shape}  (seq_len=5, d_model=8)')
print(f'Num heads     : {num_heads}')
print(f'd_head        : {d_model_mha // num_heads}  (each head operates in {d_model_mha//num_heads}D)')
print(f'Output shape  : {mha_output.shape}  (same as input)')

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for h, (ax, w) in enumerate(zip(axes, mha_weights)):
    im = ax.imshow(w, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f'Head {h+1}', fontsize=10)
    ax.set_xlabel('Key')
    ax.set_ylabel('Query')
plt.suptitle('Multi-Head Attention — Each Head Learns Different Patterns', fontsize=12)
plt.tight_layout()
plt.savefig('multihead_attention.png', dpi=80, bbox_inches='tight')
plt.show()
print('Multi-head attention visualization saved.')

### 3.3 Positional Encoding — Why Transformers Need It

Unlike RNNs, Transformers process all tokens **in parallel** — they have no inherent notion of order.

Positional encoding adds **position information** to each token embedding:

```
PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
```

**Why sinusoidal?**
- Different frequencies encode different positional scales
- The model can learn to attend to relative positions
- Works for sequences longer than those seen in training

**Interview answer**: Positional encodings give the Transformer a way to distinguish "The cat chased the dog" from "The dog chased the cat".

In [ ]:
def positional_encoding(seq_len, d_model):
    """Sinusoidal positional encoding (Vaswani et al. 2017)."""
    PE = np.zeros((seq_len, d_model))
    positions = np.arange(seq_len).reshape(-1, 1)   # (seq_len, 1)
    dims = np.arange(0, d_model, 2)                  # even dimensions

    # Compute the divisor for each dimension
    div_term = np.exp(dims * (-np.log(10000.0) / d_model))  # (d_model/2,)

    PE[:, 0::2] = np.sin(positions * div_term)   # even dims: sin
    PE[:, 1::2] = np.cos(positions * div_term)   # odd dims : cos
    return PE

seq_len_pe = 50
d_model_pe = 64
PE = positional_encoding(seq_len_pe, d_model_pe)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
im = axes[0].imshow(PE, cmap='RdBu', aspect='auto', vmin=-1, vmax=1)
axes[0].set_xlabel('Embedding Dimension', fontsize=11)
axes[0].set_ylabel('Position in Sequence', fontsize=11)
axes[0].set_title('Positional Encoding Heatmap\n(red=+1, blue=-1)', fontsize=12)
plt.colorbar(im, ax=axes[0])

# Show first 4 dimensions as sine waves
for i in range(4):
    axes[1].plot(PE[:, i], label=f'dim {i}', linewidth=1.5)
axes[1].set_xlabel('Position in Sequence', fontsize=11)
axes[1].set_ylabel('Encoding Value', fontsize=11)
axes[1].set_title('Positional Encoding — First 4 Dimensions\n(different frequencies = different scales)', fontsize=12)
axes[1].legend()
axes[1].axhline(0, color='black', linewidth=0.5, linestyle='--')

plt.suptitle('Sinusoidal Positional Encoding (Vaswani et al. 2017)', fontsize=13)
plt.tight_layout()
plt.savefig('positional_encoding.png', dpi=80, bbox_inches='tight')
plt.show()
print('Positional encoding visualization saved.')

### 3.4 The Transformer Architecture — Explained in Code

A Transformer encoder block consists of:
1. Multi-Head Self-Attention
2. Add & Norm (residual connection + layer normalization)
3. Feed-Forward Network (FFN)
4. Add & Norm again

```
TransformerEncoderBlock(X):
    attn_out = MultiHeadAttention(X)         # attend to all positions
    X = LayerNorm(X + attn_out)              # residual + normalize
    ffn_out = FFN(X)                         # position-wise transform
    X = LayerNorm(X + ffn_out)               # residual + normalize
    return X
```

**Why residual connections?** They allow gradients to flow directly through the network, enabling very deep architectures (BERT has 12 layers, GPT-3 has 96 layers).

In [ ]:
def layer_norm(x, eps=1e-6):
    """Layer Normalization: normalize across feature dimension."""
    mean = x.mean(axis=-1, keepdims=True)
    std  = x.std(axis=-1, keepdims=True)
    return (x - mean) / (std + eps)

def feed_forward(x, d_ff=None):
    """Position-wise FFN: two linear layers with ReLU."""
    d_model = x.shape[-1]
    if d_ff is None:
        d_ff = 4 * d_model  # typically 4x expansion
    np.random.seed(123)
    W1 = np.random.randn(d_model, d_ff) * 0.1
    b1 = np.zeros(d_ff)
    W2 = np.random.randn(d_ff, d_model) * 0.1
    b2 = np.zeros(d_model)
    hidden = np.maximum(0, x @ W1 + b1)   # ReLU activation
    return hidden @ W2 + b2

def transformer_encoder_block(X, num_heads=4):
    """One Transformer encoder block."""
    d_model = X.shape[-1]

    # 1. Multi-Head Self-Attention + residual
    attn_out, _ = multi_head_attention(X, num_heads=num_heads, d_model=d_model)
    X = layer_norm(X + attn_out)

    # 2. Feed-Forward Network + residual
    ffn_out = feed_forward(X)
    X = layer_norm(X + ffn_out)

    return X

# Full mini-Transformer encoder
def transformer_encoder(X, num_layers=2, num_heads=4):
    """Stack of Transformer encoder blocks with positional encoding."""
    seq_len, d_model = X.shape
    PE = positional_encoding(seq_len, d_model)
    X = X + PE      # add positional information
    for layer in range(num_layers):
        X = transformer_encoder_block(X, num_heads=num_heads)
    return X

np.random.seed(42)
seq_len_tf = 6
d_model_tf = 8

# Simulate token embeddings (normally from an embedding lookup table)
X_transformer = np.random.randn(seq_len_tf, d_model_tf)

print('=== Transformer Encoder Architecture Demo ===')
print(f'Input shape : {X_transformer.shape}  (6 tokens, 8-dim embeddings)')
X_out = transformer_encoder(X_transformer, num_layers=2, num_heads=4)
print(f'Output shape: {X_out.shape}  (same shape - contextualized embeddings)')
print('\nContextualized embeddings (first 3 tokens, first 4 dims):')
print(np.round(X_out[:3, :4], 4))
print('\nNote: Output embeddings now encode CONTEXT from entire sequence.')
print('Token 0 now knows about tokens 1-5, unlike simple word2vec embeddings.')

---
## SECTION 4 — Using Pretrained Models

In practice you rarely train Transformers from scratch. The workflow is:
1. Take a pretrained model (BERT, RoBERTa, GPT, etc.)
2. Fine-tune on your specific task
3. Evaluate and deploy

### 4.1 HuggingFace Transformers — BERT with Graceful Fallback

This section tries to load BERT via the `transformers` library. If not installed, it shows the concepts with pseudocode and sklearn equivalents.

In [ ]:
# Attempt to import transformers; fall back gracefully if unavailable
try:
    from transformers import (
        AutoTokenizer, AutoModel,
        pipeline, BertTokenizer, BertModel,
        AutoModelForSequenceClassification
    )
    import torch
    TRANSFORMERS_AVAILABLE = True
    print('transformers and torch are available!')
    print(f'  transformers version: {__import__("transformers").__version__}')
    print(f'  torch version      : {torch.__version__}')
except ImportError as e:
    TRANSFORMERS_AVAILABLE = False
    print(f'transformers/torch not installed: {e}')
    print('Showing pseudocode + sklearn equivalent.')
    print('To install: pip install transformers torch sentence-transformers')

In [ ]:
if TRANSFORMERS_AVAILABLE:
    print('=== Loading BERT tokenizer and model ===')
    print('(Downloads ~440MB on first run from HuggingFace Hub)')
    print()
    try:
        tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
        model = AutoModel.from_pretrained('bert-base-uncased')
        model.eval()

        sentence = "Transformers changed everything in natural language processing."
        inputs = tokenizer(sentence, return_tensors='pt', padding=True, truncation=True)

        print('=== Tokenization Output ===')
        print(f'Input text     : {sentence}')
        tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
        print(f'Tokens         : {tokens}')
        print(f'Token IDs      : {inputs["input_ids"][0].tolist()}')
        print(f'Attention mask : {inputs["attention_mask"][0].tolist()}')

        with torch.no_grad():
            outputs = model(**inputs)
        last_hidden = outputs.last_hidden_state  # (batch=1, seq_len, 768)
        cls_embedding = last_hidden[0, 0, :]     # [CLS] token embedding

        print(f'\n=== BERT Output Shapes ===')
        print(f'last_hidden_state : {tuple(last_hidden.shape)}  (1 batch, {last_hidden.shape[1]} tokens, 768 dims)')
        print(f'[CLS] embedding   : {tuple(cls_embedding.shape)}  (sentence representation)')
        print(f'[CLS] embedding (first 10 dims): {cls_embedding[:10].numpy().round(4)}')

    except Exception as e:
        print(f'Error loading BERT: {e}')
        TRANSFORMERS_AVAILABLE = False

if not TRANSFORMERS_AVAILABLE:
    print('=== BERT Concepts (No transformers installed) ===')
    print()
    print('BERT TOKENIZATION PSEUDOCODE:')
    print('-' * 50)
    print('tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")')
    print('inputs = tokenizer(text, return_tensors="pt")')
    print('# Returns: input_ids, attention_mask, token_type_ids')
    print()
    print('BERT EMBEDDING PSEUDOCODE:')
    print('-' * 50)
    print('model = BertModel.from_pretrained("bert-base-uncased")')
    print('outputs = model(**inputs)')
    print('cls_embedding = outputs.last_hidden_state[:, 0, :]  # (batch, 768)')
    print()
    print('SKLEARN EQUIVALENT: TF-IDF gives sparse bag-of-words representations.')
    print('BERT gives dense 768-dim contextual representations.')
    print('Key difference: BERT encodes context ("bank" in finance vs. river).')


### 4.2 Sentence Embeddings with Sentence-Transformers

Sentence-transformers produce fixed-size embeddings optimized for semantic similarity.

**Mean pooling vs [CLS] token:**
- Raw BERT [CLS] was pretrained for MLM/NSP, not similarity
- Sentence-BERT fine-tunes with contrastive learning for semantic tasks
- Mean pooling over all tokens often outperforms [CLS] for similarity

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    SBERT_AVAILABLE = True
    print('sentence-transformers available!')
except ImportError:
    SBERT_AVAILABLE = False
    print('sentence-transformers not installed.')
    print('To install: pip install sentence-transformers')

if SBERT_AVAILABLE:
    print('Loading all-MiniLM-L6-v2 (fast, ~80MB)...')
    try:
        sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

        sentences_sbert = [
            "Machine learning is a subset of AI",
            "Deep learning uses neural networks",
            "The cat sat on the mat",
            "A dog lay on the rug",
            "Natural language processing handles text"
        ]

        embeddings = sbert_model.encode(sentences_sbert)
        print(f'Embedding shape: {embeddings.shape}  ({len(sentences_sbert)} sentences, 384 dims)')

        # Cosine similarity matrix
        from sklearn.metrics.pairwise import cosine_similarity as cos_sim
        sim_matrix = cos_sim(embeddings)

        fig, ax = plt.subplots(figsize=(8, 6))
        im = ax.imshow(sim_matrix, cmap='YlOrRd', vmin=0, vmax=1)
        plt.colorbar(im, ax=ax)
        ax.set_xticks(range(len(sentences_sbert)))
        ax.set_yticks(range(len(sentences_sbert)))
        short_labels = [s[:25] + '...' if len(s) > 25 else s for s in sentences_sbert]
        ax.set_xticklabels(short_labels, rotation=30, ha='right', fontsize=8)
        ax.set_yticklabels(short_labels, fontsize=8)
        ax.set_title('Sentence-BERT Semantic Similarity Matrix', fontsize=12)
        for i in range(len(sentences_sbert)):
            for j in range(len(sentences_sbert)):
                ax.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=9)
        plt.tight_layout()
        plt.savefig('sbert_similarity.png', dpi=80, bbox_inches='tight')
        plt.show()
        print('Semantic similarity matrix saved.')
    except Exception as e:
        print(f'Error with sentence-transformers: {e}')
        SBERT_AVAILABLE = False

if not SBERT_AVAILABLE:
    print()
    print('=== Sentence Embeddings — Concept + sklearn Approximation ===')
    print()
    print('SENTENCE-BERT PSEUDOCODE:')
    print('  model = SentenceTransformer("all-MiniLM-L6-v2")')
    print('  embeddings = model.encode(sentences)  # (N, 384)')
    print('  # Use cosine similarity for semantic search')
    print()
    print('SKLEARN APPROXIMATION (bag of words similarity):')
    sentences_demo = [
        'Machine learning is a subset of AI',
        'Deep learning uses neural networks',
        'The cat sat on the mat',
        'A dog lay on the rug',
        'Natural language processing handles text'
    ]
    tfidf_demo = TfidfVectorizer()
    X_demo = tfidf_demo.fit_transform(sentences_demo)
    from sklearn.metrics.pairwise import cosine_similarity as cos_sim_sk
    sim_matrix_demo = cos_sim_sk(X_demo)
    print('TF-IDF cosine similarity matrix:')
    for i, s in enumerate(sentences_demo):
        row = f'  {s[:25]+"..." if len(s)>25 else s:<28}: '
        row += '  '.join(f'{sim_matrix_demo[i,j]:.2f}' for j in range(len(sentences_demo)))
        print(row)

### 4.3 Fine-Tuning BERT for Classification

**Fine-tuning workflow:**
1. Load pretrained BERT
2. Add a classification head on top of [CLS]
3. Train with a small learning rate (2e-5 to 5e-5)
4. Train for 2-5 epochs

**Why small learning rate?** The pretrained weights are already good. Large updates would destroy the learned representations.

In [ ]:
if TRANSFORMERS_AVAILABLE:
    print('=== BERT Fine-Tuning for Text Classification ===')
    try:
        from transformers import AutoModelForSequenceClassification
        from torch.optim import AdamW

        # Load BERT with classification head
        num_labels = 4  # our 4 newsgroup categories
        bert_clf = AutoModelForSequenceClassification.from_pretrained(
            'bert-base-uncased',
            num_labels=num_labels
        )

        print(f'Model: bert-base-uncased + classification head')
        print(f'Total parameters: {sum(p.numel() for p in bert_clf.parameters()):,}')
        print(f'Trainable params: {sum(p.numel() for p in bert_clf.parameters() if p.requires_grad):,}')
        print()
        print('Fine-tuning setup:')
        print('  optimizer = AdamW(model.parameters(), lr=2e-5)')
        print('  scheduler = linear warmup over first 10% of steps')
        print('  epochs = 3')
        print('  batch_size = 16')
        print()
        print('Training loop pseudocode:')
        print('  for epoch in range(3):')
        print('      for batch in dataloader:')
        print('          outputs = model(**batch)')
        print('          loss = outputs.loss')
        print('          loss.backward()')
        print('          optimizer.step()')
        print('          scheduler.step()')
        print('          optimizer.zero_grad()')
        print()
        print('Note: Full fine-tuning requires GPU and ~30 min for 3 epochs on 20newsgroups')
    except Exception as e:
        print(f'BERT fine-tuning setup error: {e}')
else:
    print('=== BERT Fine-Tuning (Conceptual Overview — no transformers installed) ===')
    print()
    print('ARCHITECTURE:')
    print('  Input text')
    print('  -> BERT Tokenizer (WordPiece)')
    print('  -> BERT Encoder (12 layers of attention)')
    print('  -> [CLS] token representation (768 dims)')
    print('  -> Linear(768, num_classes)')
    print('  -> Softmax -> class probabilities')
    print()
    print('KEY HYPERPARAMETERS:')
    params = [
        ('Learning rate', '2e-5 to 5e-5 (much lower than training from scratch)'),
        ('Batch size', '16 or 32'),
        ('Epochs', '2-5 (more causes overfitting on small datasets)'),
        ('Warmup steps', '10% of total steps'),
        ('Weight decay', '0.01 (L2 regularization)'),
    ]
    for name, val in params:
        print(f'  {name:<20}: {val}')
    print()
    print('SKLEARN EQUIVALENT PIPELINE (when BERT unavailable):')
    print('  TfidfVectorizer(max_features=50000, ngram_range=(1,2))')
    print('  + LogisticRegression(C=1.0) or LinearSVC()')
    print('  Typical 20newsgroups accuracy: ~85-90% vs BERT ~95%')

### 4.4 Zero-Shot Classification

Zero-shot classification lets you classify text into categories **without any training examples** for those categories.

**How it works:**
- Uses a Natural Language Inference (NLI) model
- Frames classification as: "Does this text entail the hypothesis: 'This is about sports'?"
- Works surprisingly well for many business use cases

In [ ]:
if TRANSFORMERS_AVAILABLE:
    print('=== Zero-Shot Classification with HuggingFace pipeline ===')
    try:
        zs_classifier = pipeline(
            'zero-shot-classification',
            model='facebook/bart-large-mnli'
        )
        texts = [
            "The quarterly earnings report showed a 15% revenue increase",
            "The team scored three goals in the final minute to win",
            "Scientists discovered a new exoplanet in the habitable zone"
        ]
        candidate_labels = ['finance', 'sports', 'science', 'politics', 'entertainment']

        for text in texts:
            result = zs_classifier(text, candidate_labels)
            print(f'Text: {text[:60]}...')
            print(f'Top prediction: {result["labels"][0]} ({result["scores"][0]:.1%})')
            print()
    except Exception as e:
        print(f'Zero-shot pipeline error: {e}')
else:
    print('=== Zero-Shot Classification (Conceptual) ===')
    print()
    print('PSEUDOCODE:')
    print('  from transformers import pipeline')
    print('  classifier = pipeline("zero-shot-classification",')
    print('                        model="facebook/bart-large-mnli")')
    print()
    print('  result = classifier(')
    print('      "The stock market crashed today",')
    print('      candidate_labels=["finance", "sports", "science"]')
    print('  )')
    print('  # result["labels"][0] -> "finance"  (no training needed!)')
    print()
    print('SKLEARN APPROACH (requires labeled examples):')
    # Demonstrate with our trained newsgroup classifier
    test_texts = [
        "The hockey puck hit the goal post hard",
        "NASA launched a new satellite into orbit today",
        "The vertex shader in OpenGL renders the polygon",
        "The gun control bill passed the senate vote"
    ]
    category_names = categories
    print('  Using our TF-IDF + LR classifier (requires training data):')
    for txt in test_texts:
        pred = text_clf_pipeline.predict([txt])[0]
        proba = text_clf_pipeline.predict_proba([txt])[0]
        print(f'  [{category_names[pred]:22s} {max(proba):.0%}] {txt}')

---
## SECTION 5 — 15 NLP Interview Questions with Answers

These are real questions asked at top tech companies. Each has a plain English answer plus code that proves the point.

### Q1: What is the difference between Word2Vec CBOW and Skip-gram?

**Answer:**
- **CBOW** (Continuous Bag of Words): predicts the *center* word from surrounding context words. Trains faster, better for frequent words.
- **Skip-gram**: predicts *surrounding context* words from the center word. Slower but better for rare words and larger datasets.

**Rule of thumb:** Skip-gram for smaller datasets, CBOW when you have abundant data and need speed.

In [ ]:
# Illustrate CBOW vs Skip-gram with a toy example
sentence = ['the', 'quick', 'brown', 'fox', 'jumps']
window = 2

print('=== CBOW Training Examples (context -> center) ===')
cbow_pairs = []
for i in range(len(sentence)):
    context = []
    for j in range(max(0, i-window), min(len(sentence), i+window+1)):
        if j != i:
            context.append(sentence[j])
    if context:
        cbow_pairs.append((context, sentence[i]))
        print(f'  Context: {str(context):40s} -> Predict: "{sentence[i]}"')

print()
print('=== Skip-gram Training Examples (center -> context) ===')
for i in range(len(sentence)):
    for j in range(max(0, i-window), min(len(sentence), i+window+1)):
        if j != i:
            print(f'  Center: "{sentence[i]:10s}" -> Predict context: "{sentence[j]}"')

print()
print('Key insight:')
print('  CBOW: fewer training samples (one per center word)')
print('  Skip-gram: more training samples (window*2 per center word)')
print('  Skip-gram trains rare words better because they appear as targets more often')

### Q2: Why can't RNNs handle long sequences well?

**Answer:** Two reasons:

1. **Vanishing gradients**: In backpropagation through time, gradients are multiplied by weights at each time step. If weights < 1, gradients shrink exponentially. After 50+ steps, gradients are nearly zero — the network cannot learn long-range dependencies.

2. **Bottleneck**: The entire sequence history must be compressed into a single fixed-size hidden state vector. Information about early tokens is overwritten by later ones.

**LSTM/GRU** mitigate vanishing gradients with gating mechanisms. **Attention** solves the bottleneck by allowing direct access to all past states.

In [ ]:
# Demonstrate vanishing gradients numerically
np.random.seed(42)

# Simulate gradient flow through T time steps
def simulate_gradient_flow(T=100, weight=0.8):
    """Track how much gradient remains from step T back to step 0."""
    gradient = 1.0  # gradient at the last step
    gradients = [gradient]
    for t in range(T):
        gradient *= weight  # multiply by recurrent weight
        gradients.append(gradient)
    return gradients

T = 50
grad_08 = simulate_gradient_flow(T, weight=0.8)
grad_09 = simulate_gradient_flow(T, weight=0.9)
grad_10 = simulate_gradient_flow(T, weight=1.0)
grad_11 = simulate_gradient_flow(T, weight=1.1)  # exploding

fig, ax = plt.subplots(figsize=(10, 5))
steps = range(T + 1)
ax.semilogy(steps, grad_08, label='w=0.8 (vanishing)', color='red', linewidth=2)
ax.semilogy(steps, grad_09, label='w=0.9 (slow vanish)', color='orange', linewidth=2)
ax.semilogy(steps, grad_10, label='w=1.0 (stable)', color='green', linewidth=2)
ax.semilogy(steps, grad_11, label='w=1.1 (exploding)', color='purple', linewidth=2, linestyle='--')
ax.set_xlabel('Steps back in time', fontsize=12)
ax.set_ylabel('Gradient magnitude (log scale)', fontsize=12)
ax.set_title('Vanishing/Exploding Gradients in RNNs\n(gradient after backpropagating through T steps)', fontsize=13)
ax.legend(fontsize=11)
ax.axhline(y=0.001, color='black', linestyle=':', alpha=0.5)
ax.text(T*0.7, 0.0015, 'threshold ~ 0.001', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('vanishing_gradients.png', dpi=80, bbox_inches='tight')
plt.show()

print(f'With w=0.8, gradient at step 50: {grad_08[50]:.2e}')
print(f'With w=0.9, gradient at step 50: {grad_09[50]:.2e}')
print(f'With w=1.0, gradient at step 50: {grad_10[50]:.2e}')
print('\nConclusion: w=0.8 makes gradient disappear after ~30 steps.')
print('Attention bypasses this by computing direct connections between all positions.')

### Q3: What problem does attention solve?

**Answer:** Attention solves two problems:

1. **The information bottleneck**: RNNs compress the entire context into one fixed-size vector. Attention gives the decoder *direct access* to all encoder hidden states.

2. **Long-range dependency**: Attention creates *direct connections* between any two positions in O(1) operations, regardless of distance. RNNs need O(n) operations to relate position 1 to position n.

**Proof**: Machine translation with attention: "The animal didn't cross the street because *it* was too tired" — attention correctly links "it" to "animal" regardless of distance.

In [ ]:
# Visualize the coreference resolution with attention
words_coreref = ['The', 'animal', 'did', 'not', 'cross', 'the', 'street', 'because', 'it', 'was', 'tired']

# Simulate attention from 'it' to all other words
# 'it' should attend most strongly to 'animal'
np.random.seed(0)
base_attn = np.random.rand(len(words_coreref)) * 0.05
base_attn[1] = 0.55   # 'animal' gets high attention
base_attn[8] = 0.20   # 'it' attends to itself somewhat
base_attn[6] = 0.10   # slight attention to 'street' (distractor)
base_attn = base_attn / base_attn.sum()

fig, ax = plt.subplots(figsize=(12, 3))
colors = ['#d62728' if a > 0.2 else '#aec7e8' if a > 0.08 else '#e0e0e0'
          for a in base_attn]
bars = ax.bar(range(len(words_coreref)), base_attn, color=colors, edgecolor='black')
ax.set_xticks(range(len(words_coreref)))
ax.set_xticklabels(words_coreref, fontsize=11)
ax.set_ylabel('Attention weight', fontsize=11)
ax.set_title('Attention from token "it" to all other tokens\n(red = high attention, correctly resolves "it" -> "animal")', fontsize=12)
for i, (bar, a) in enumerate(zip(bars, base_attn)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{a:.2f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig('coreference_attention.png', dpi=80, bbox_inches='tight')
plt.show()
print('Coreference attention visualization saved.')

### Q4: What is BERT and how is it different from GPT?

**Answer:**

| Aspect | BERT | GPT |
|---|---|---|
| Architecture | Encoder only | Decoder only |
| Training objective | Masked Language Model (MLM) + NSP | Next token prediction (causal LM) |
| Attention | Bidirectional (sees all tokens) | Causal (only sees past tokens) |
| Best for | Understanding tasks (classification, QA) | Generation tasks (text, code, chat) |
| Example models | BERT, RoBERTa, ALBERT | GPT-2, GPT-3, GPT-4 |

**Key insight**: BERT uses **[MASK]** tokens during training — it sees both left and right context simultaneously. GPT is **autoregressive** — it only sees past context, enabling generation.

In [ ]:
# Illustrate BERT MLM vs GPT causal LM training objectives
sentence_mlm = ['The', '[MASK]', 'sat', 'on', 'the', 'mat']
# BERT predicts: what word fills [MASK]? (has full context: 'The __ sat on the mat')

sentence_causal = ['The', 'cat', 'sat', 'on', 'the', 'mat']
# GPT predicts each word from only previous words

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# BERT: bidirectional attention (all attend to all, except [MASK] predicts itself)
bert_mask = np.ones((6, 6))  # bidirectional = full attention
bert_mask[1, 1] = 0.5  # [MASK] position

im1 = ax1.imshow(bert_mask, cmap='YlGn', vmin=0, vmax=1)
ax1.set_xticks(range(6))
ax1.set_yticks(range(6))
ax1.set_xticklabels(sentence_mlm, fontsize=10)
ax1.set_yticklabels(sentence_mlm, fontsize=10)
ax1.set_title('BERT: Bidirectional Attention\n(every token sees ALL tokens)', fontsize=11)
ax1.set_xlabel('Key (attends to)')
ax1.set_ylabel('Query (attending from)')
plt.colorbar(im1, ax=ax1, fraction=0.046)

# GPT: causal (lower triangular) attention mask
gpt_mask = np.tril(np.ones((6, 6)))

im2 = ax2.imshow(gpt_mask, cmap='YlOrRd', vmin=0, vmax=1)
ax2.set_xticks(range(6))
ax2.set_yticks(range(6))
ax2.set_xticklabels(sentence_causal, fontsize=10)
ax2.set_yticklabels(sentence_causal, fontsize=10)
ax2.set_title('GPT: Causal (Autoregressive) Attention\n(each token sees ONLY past tokens)', fontsize=11)
ax2.set_xlabel('Key (attends to)')
ax2.set_ylabel('Query (attending from)')
plt.colorbar(im2, ax=ax2, fraction=0.046)

plt.suptitle('BERT vs GPT Attention Patterns', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('bert_vs_gpt.png', dpi=80, bbox_inches='tight')
plt.show()
print('BERT vs GPT attention pattern visualization saved.')

### Q5: How would you handle out-of-vocabulary (OOV) words?

**Answer:** Several strategies, in order of sophistication:

1. **Replace with <UNK> token** — simple but loses all information
2. **Character-level model** — build from characters, no OOV possible
3. **Subword tokenization (BPE/WordPiece)** — "unhappiness" -> "un", "##happiness" -> still meaningful pieces
4. **fastText** — uses character n-grams in embeddings, handles OOV by summing subword vectors
5. **Domain-specific preprocessing** — normalize ("4evr" -> "forever"), spell-correct

**Interview trap**: Don't just say "replace with UNK". Mention subword tokenization since it's how all modern systems work.

In [ ]:
# Demonstrate subword tokenization logic (simplified BPE)
def byte_pair_encode_demo(word, vocab):
    """
    Simplified BPE: split word into known subword pieces.
    Falls back to character-level if subword not in vocab.
    """
    if word in vocab:
        return [word]

    # Try to find longest matching prefix in vocab
    result = []
    remaining = word
    while remaining:
        found = False
        for length in range(len(remaining), 0, -1):
            subword = remaining[:length]
            if subword in vocab or length == 1:
                if length == 1 and subword not in vocab:
                    result.append('[UNK_CHAR:' + subword + ']')
                else:
                    result.append(subword if length > 1 else subword)
                remaining = remaining[length:]
                found = True
                break
        if not found:
            result.append('[UNK]')
            break
    return result

# Simulated BERT-style vocabulary (simplified)
bert_vocab = {
    'un', 'happy', 'happiness', 'unhappy', 'the', 'cat', 'sat',
    'neural', 'network', 'trans', 'former', 'transformer',
    'learn', 'learning', 'machine', 'deep', '##ing', '##er',
    'pro', 'cess', 'ing', 'process', 'processing', 'natural',
    'lang', 'language', 'text', 'classi', 'fication', 'classification'
}

test_oov_words = [
    'transformer',      # in vocab
    'unhappiness',      # needs decomposition: un + happiness
    'xyzabc123',        # truly OOV -- unusual word
    'preprocessing',    # in vocab
    'languagemodel',    # compound: language + model
]

print('=== OOV Handling: Subword Tokenization Demo ===')
print(f'{"Word":<20} {"Subword Pieces"}')
print('-' * 60)
for word in test_oov_words:
    pieces = byte_pair_encode_demo(word, bert_vocab)
    oov_flag = ' ** OOV **' if word not in bert_vocab else ''
    print(f'{word:<20} {pieces}{oov_flag}')

print()
print('Real BERT tokenizer would produce WordPiece tokens like:')
print('  "unhappiness" -> ["un", "##happ", "##iness"]')
print('  "transformers" -> ["transform", "##ers"]')
print('  Key: ## prefix means "continuation of previous subword"')

### Q6-Q10: Rapid-Fire Interview Questions

In [ ]:
print('Q6: What is the computational complexity of self-attention?')
print('A: O(n^2 * d) where n=sequence length, d=dimension.')
print('   Each token attends to all n tokens -> n^2 attention scores.')
print('   This is why long documents are expensive and why models like Longformer')
print('   use sparse attention patterns (O(n) instead of O(n^2)).')
print()
print('Q7: What is the difference between encoder-only, decoder-only, and encoder-decoder?')
print('A: Encoder-only (BERT): bidirectional context, best for understanding tasks.')
print('   Decoder-only (GPT): autoregressive, best for generation.')
print('   Encoder-decoder (T5, BART): encode input, decode output.')
print('   Use case: translation, summarization, question answering.')
print()
print('Q8: What is layer normalization and why use it in Transformers?')
print('A: Normalize across feature dimension (not batch dimension).')
print('   Formula: (x - mean) / (std + eps), then scale+shift.')
print('   Why transformers: batch size varies, sequences vary in length.')
print('   LayerNorm works per-sample; BatchNorm needs large batches.')
print()
print('Q9: What is teacher forcing in sequence-to-sequence models?')
print('A: During training, feed the true previous output as decoder input.')
print('   Without it: errors cascade (wrong predicted token -> wrong next prediction).')
print('   Downside: train-test mismatch (inference uses predicted tokens).')
print('   Solutions: scheduled sampling (gradually reduce teacher forcing).')
print()
print('Q10: What is perplexity and what does it measure in language models?')
print('A: Perplexity = 2^(average bits per word) = exp(cross-entropy loss).')
print('   Low perplexity = model assigns high probability to actual text = better model.')
print('   GPT-2 perplexity on Penn Treebank: ~35. Human: ~18.')
print('   Caveat: perplexity is corpus-specific. Cannot compare across datasets.')

In [ ]:
# Q10 proof: compute perplexity
def compute_perplexity(probabilities):
    """
    Compute perplexity from a list of token probabilities.
    perplexity = exp(-1/N * sum(log(p_i)))
    """
    log_probs = [np.log(p + 1e-10) for p in probabilities]
    avg_log_prob = np.mean(log_probs)
    return np.exp(-avg_log_prob)

# Simulate language model probabilities on a short sequence
np.random.seed(42)

# Good model: assigns high probability to correct tokens
good_model_probs   = [0.85, 0.72, 0.91, 0.68, 0.77, 0.83, 0.70, 0.88]
# Poor model: assigns lower probability
poor_model_probs   = [0.30, 0.25, 0.35, 0.20, 0.28, 0.31, 0.22, 0.34]
# Random model: uniform probability over 50000 vocab
random_model_probs = [1/50000] * 8

print('=== Perplexity Comparison ===')
for name, probs in [
    ('Good language model  ', good_model_probs),
    ('Poor language model  ', poor_model_probs),
    ('Random (50k vocab)   ', random_model_probs)
]:
    ppl = compute_perplexity(probs)
    avg_p = np.mean(probs)
    print(f'{name}: avg_prob={avg_p:.3f}, perplexity={ppl:.1f}')

print()
print('Lower perplexity = better. Random model perplexity = vocab_size = 50,000.')

### Q11-Q15: Advanced NLP Interview Questions

In [ ]:
print('Q11: How does BERT handle subword tokenization?')
print('A: BERT uses WordPiece tokenization:')
print('   1. Start with a character-level vocabulary.')
print('   2. Merge frequent character pairs iteratively.')
print('   3. "## " prefix marks continuation subwords.')
print('   4. Special tokens: [CLS] (classification), [SEP] (separator), [MASK], [PAD].')
print('   Result: 30,000 token vocabulary covers any English text.')
print()
print('Q12: What is the difference between fine-tuning and prompt engineering?')
print('A: Fine-tuning: update model weights with task-specific labeled data.')
print('   Pros: highest accuracy; Cons: expensive, requires labeled data.')
print('   Prompt engineering: craft the input to elicit desired behavior, NO weight updates.')
print('   Pros: no training needed; Cons: less reliable, model-dependent.')
print('   Continuum: zero-shot -> few-shot -> prompt tuning -> fine-tuning.')
print()
print('Q13: How would you build a semantic search system?')
print('A: Two-stage pipeline:')
print('   Stage 1 (retrieval): encode documents with sentence-transformer,')
print('                        store in FAISS vector index.')
print('   Stage 2 (re-rank) : use cross-encoder model for exact similarity.')
print('   Query time: encode query -> find top-K from FAISS -> re-rank top-K.')
print('   This is how RAG (Retrieval Augmented Generation) works.')
print()
print('Q14: What is the softmax temperature and why does it matter?')
print('A: Temperature T scales logits before softmax:')
print('   p_i = exp(z_i / T) / sum(exp(z_j / T))')
print('   T < 1: sharpens distribution (more confident, less diverse outputs)')
print('   T = 1: standard softmax')
print('   T > 1: flattens distribution (more diverse, more creative outputs)')
print('   Used in: language model sampling, knowledge distillation.')
print()
print('Q15: What are the limitations of BERT?')
print('A: Key limitations:')
print('   1. Fixed max sequence length (512 tokens for base).')
print('   2. Expensive: 110M parameters, slow for real-time inference.')
print('   3. Static embeddings after fine-tuning (not updated at inference time).')
print('   4. MLM pretraining creates artificial [MASK] tokens not seen at inference.')
print('   5. Primarily English; multilingual BERT loses performance per-language.')
print('   Solutions: DistilBERT (60% smaller), ALBERT, RoBERTa, domain-specific models.')

In [ ]:
# Q14 code proof: temperature effect on softmax
def softmax_temperature(logits, T=1.0):
    scaled = np.array(logits) / T
    scaled -= scaled.max()
    exp_scaled = np.exp(scaled)
    return exp_scaled / exp_scaled.sum()

logits = [3.0, 2.0, 1.0, 0.5, 0.1]  # logits for 5 vocabulary tokens
temperatures = [0.3, 0.7, 1.0, 2.0, 5.0]

print('=== Softmax Temperature Effect ===')
print(f'Logits: {logits}')
print()
print(f'{"Temperature":>12} | {" ".join(f"p{i}"  for i in range(5)):40s} | {"Entropy":>8} | {"Character":>15}')
print('-' * 90)
for T in temperatures:
    probs = softmax_temperature(logits, T)
    entropy = -np.sum(probs * np.log(probs + 1e-10))
    char = 'sharp/confident' if T < 0.7 else ('uniform/diverse' if T > 1.5 else 'balanced')
    probs_str = '  '.join(f'{p:.3f}' for p in probs)
    print(f'{T:>12.1f} | {probs_str:40s} | {entropy:>8.3f} | {char:>15}')

fig, axes = plt.subplots(1, len(temperatures), figsize=(14, 4))
for ax, T in zip(axes, temperatures):
    probs = softmax_temperature(logits, T)
    ax.bar(range(5), probs, color='steelblue', alpha=0.8)
    ax.set_title(f'T={T}', fontsize=11)
    ax.set_xticks(range(5))
    ax.set_xticklabels([f'w{i}' for i in range(5)])
    ax.set_ylim(0, 1)
    if T == temperatures[0]:
        ax.set_ylabel('Probability')
plt.suptitle('Softmax Temperature — Low T=Confident, High T=Diverse', fontsize=12)
plt.tight_layout()
plt.savefig('temperature_softmax.png', dpi=80, bbox_inches='tight')
plt.show()
print('Temperature visualization saved.')

---
## Summary & Quick Reference

This notebook covered the complete NLP interview spectrum:

### Preprocessing
- Tokenization: word -> subword (BPE/WordPiece) for modern models
- Stemming vs Lemmatization: use lemmatization for semantics
- Stop words: **keep for sentiment** (negation), remove for IR/TF-IDF
- TF-IDF: TF * IDF, downweights common words

### Classical Models
- TF-IDF + Logistic Regression: strong baseline, interpret weights
- LDA: unsupervised topics, soft assignment per document
- Metrics: precision (false alarm cost), recall (miss cost), F1 (balance)

### Attention & Transformers
- Self-attention: Q/K/V projection, scaled dot product, softmax
- Multi-head: parallel heads learn different relationship types
- Positional encoding: sinusoidal, enables position awareness
- Residual + LayerNorm: enables deep architectures

### Modern Models
- BERT: bidirectional encoder, MLM pretraining, fine-tune for understanding
- GPT: causal decoder, autoregressive, for generation
- Sentence-BERT: semantic similarity, dense retrieval
- Zero-shot: NLI framing, no training data needed

### Key Numbers to Remember
- BERT-base: 12 layers, 12 heads, 768 dims, 110M params
- BERT max sequence: 512 tokens
- Standard fine-tuning LR: 2e-5 to 5e-5
- Typical fine-tuning epochs: 2-5

### Interview Cheat Sheet
1. Always start with TF-IDF + LR baseline
2. BERT is bidirectional (encoder), GPT is causal (decoder)
3. Attention is O(n^2), RNNs are O(n) per step
4. Vanishing gradients: why LSTMs and attention exist
5. Subword tokenization solves OOV for modern models

In [ ]:
print('=' * 60)
print('NOTEBOOK COMPLETE: NLP from TF-IDF to Transformers')
print('=' * 60)
print()
print('Sections covered:')
print('  1. Text Preprocessing (tokenization, TF-IDF, Word2Vec)')
print('  2. Classical NLP (classification, sentiment, NER, LDA)')
print('  3. Attention Mechanism (self-attention, multi-head, PE)')
print('  4. Pretrained Models (BERT, Sentence-BERT, zero-shot)')
print('  5. 15 Interview Questions with Code Proofs')
print()
print('Visualizations generated:')
images = [
    'confusion_matrix.png',
    'lda_topics.png',
    'attention_heatmap.png',
    'multihead_attention.png',
    'positional_encoding.png',
    'vanishing_gradients.png',
    'bert_vs_gpt.png',
    'coreference_attention.png',
    'temperature_softmax.png'
]
for img in images:
    print(f'  - {img}')
print()
print('Next steps:')
print('  - Run with transformers/torch installed for live BERT demos')
print('  - Practice implementing attention from memory (common whiteboard Q)')
print('  - Explore HuggingFace Model Hub: huggingface.co/models')